# 3. Classify (LLM + pgvector)

Classifies every invoice for every schema with one method: embed the
invoice, retrieve the most similar examples from pgvector (notebook 2),
and ask the LLM — through **Unity Gateway** — to pick a code from those
candidates. This scales to any taxonomy size because the model only ever
sees a handful of candidates, never the whole tree.

Each invoice becomes one row in `predictions` with `source='llm'`,
a 0-1 `confidence`, the model's `rationale`, the retrieved `candidates`,
and a `status`/`reason` so unclassified invoices explain themselves. Every
call is wrapped in an **MLflow trace** (`trace_id` on the row) so you can
see exactly what was retrieved and what the model replied.

**Adding a new schema?** Subclass `ClassificationSchema`, register it, then
re-run notebooks 0 → 2 → 3. No edits here.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openai openpyxl pydantic 'databricks-sdk>=0.50.0' psycopg2-binary mlflow
%restart_python

In [ ]:
import os, sys, uuid
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import list_schemas
from src.categorize import classify_with_llm
from src.llm import gateway_client

spark = get_spark()
config = load_config()
client = gateway_client()   # Unity Gateway

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('invoices_table', config.invoices)
dbutils.widgets.text('lakebase_instance', config.lakebase_instance)
dbutils.widgets.text('llm_endpoint', config.large_llm_endpoint)
dbutils.widgets.text('embed_endpoint', 'databricks-gte-large-en')
dbutils.widgets.text('limit_rows', '0')   # 0 = all; set small for a fast/cheap run

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
invoices_table = dbutils.widgets.get('invoices_table')
lakebase_instance = dbutils.widgets.get('lakebase_instance')
llm_endpoint = dbutils.widgets.get('llm_endpoint')
embed_endpoint = dbutils.widgets.get('embed_endpoint')
limit_rows = int(dbutils.widgets.get('limit_rows'))
spark.sql(f'USE {catalog}.{schema}')

## Lakebase connection

The classifier retrieves similar examples from the pgvector `examples`
table built in notebook 2. `pg_connect` returns a fresh psycopg2
connection from a short-lived Lakebase credential.

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg2

w = WorkspaceClient()

def pg_connect():
    inst = w.database.get_database_instance(name=lakebase_instance)
    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()), instance_names=[lakebase_instance]
    )
    me = w.current_user.me()
    user = me.emails[0].value if me.emails else me.user_name
    return psycopg2.connect(
        host=inst.read_write_dns, dbname=config.lakebase_dbname,
        user=user, password=cred.token, sslmode='require',
    )

## Classify every schema

`classify_with_llm` embeds the invoices, retrieves candidates from
pgvector, calls the LLM, and MERGEs the results into `predictions` with
`source='llm'`. One call per schema.

In [ ]:
for s in list_schemas():
    print(f'\n==> {s.name}')
    n = classify_with_llm(
        spark, s,
        catalog=catalog, schema_db=schema,
        pg_connect=pg_connect,
        client=client,
        chat_endpoint=llm_endpoint,
        embed_endpoint=embed_endpoint,
        invoices_table=invoices_table,
        top_k=s.spec.top_k,
        limit_rows=limit_rows,
    )
    print(f'    wrote {n:,} predictions')

## Sanity check

In [ ]:
spark.sql(f'''
SELECT schema_name, source, status, reason,
       COUNT(*) AS n,
       ROUND(AVG(confidence), 3) AS avg_conf
FROM {catalog}.{schema}.predictions
GROUP BY schema_name, source, status, reason
ORDER BY schema_name, source, n DESC
''').display()